# Module 5 — 矩陣分解與工程分析

> **對應程度**：大學線代核心，工程數值方法基礎

本模組涵蓋：
1. LU 分解
2. QR 分解
3. Cholesky 分解
4. SVD — 奇異值分解（核心）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg as sla
import time
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.linalg_utils import (
    lu_decomposition, qr_decomposition, cholesky_decomposition
)
from src.physics_models import heat_conduction_1d
from src.visualizer import set_style, plot_matrix_heatmap

set_style()
print('Module 5 載入完成！')

---
## 5.1 LU 分解

$A = LU$（下三角 × 上三角）

### 物理應用：有限元素法反覆求解（同一剛度矩陣，不同荷載）

In [ ]:
# LU 分解基礎
A = np.array([[2, -1, 0], [-1, 2, -1], [0, -1, 2]], dtype=float)
L, U = lu_decomposition(A)

print('A =')
print(A)
print('\nL =')
print(np.round(L, 4))
print('\nU =')
print(np.round(U, 4))
print(f'\n重組誤差: {np.linalg.norm(L @ U - A):.2e} ✓')

In [ ]:
# 有限元素法：LU 分解只做一次，解多組荷載
n = 5
K = np.diag(2*np.ones(n)) + np.diag(-np.ones(n-1), 1) + np.diag(-np.ones(n-1), -1)
L, U = lu_decomposition(K)

# 三種荷載
loads = {
    '集中力（中間）': np.array([0, 0, 10, 0, 0]),
    '均佈力': np.array([2, 2, 2, 2, 2]),
    '三角形荷載': np.array([1, 2, 3, 2, 1]),
}

fig, ax = plt.subplots(figsize=(10, 6))
for name, f in loads.items():
    # 前向代換 Ly = f
    y = sla.solve_triangular(L, f, lower=True)
    # 回代 Ux = y
    x = sla.solve_triangular(U, y, lower=False)
    ax.plot(range(n), x, 'o-', lw=2, label=name)
    print(f'{name}: 最大位移 = {np.max(np.abs(x)):.4f}')

ax.set_xlabel('節點')
ax.set_ylabel('位移')
ax.set_title('LU 分解解多組荷載')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# LU vs 直接求逆效率比較
sizes = [50, 100, 200, 500]
times_lu = []
times_inv = []
n_rhs = 10  # 10 組右手邊

for n in sizes:
    A = np.random.default_rng(42).normal(size=(n, n))
    A = A.T @ A + n * np.eye(n)  # 確保正定
    bs = [np.random.default_rng(i).normal(size=n) for i in range(n_rhs)]

    # LU 方法
    t0 = time.perf_counter()
    lu, piv = sla.lu_factor(A)
    for b in bs:
        sla.lu_solve((lu, piv), b)
    times_lu.append(time.perf_counter() - t0)

    # 求逆方法
    t0 = time.perf_counter()
    A_inv = np.linalg.inv(A)
    for b in bs:
        A_inv @ b
    times_inv.append(time.perf_counter() - t0)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sizes, times_lu, 'bo-', lw=2, label='LU 分解')
ax.plot(sizes, times_inv, 'rs-', lw=2, label='求逆矩陣')
ax.set_xlabel('矩陣大小 n')
ax.set_ylabel('時間 (秒)')
ax.set_title(f'LU 分解 vs 求逆（{n_rhs} 組右手邊）')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

---
## 5.2 QR 分解

$A = QR$（正交 × 上三角）

### 物理應用：旋轉矩陣正交化（慣性導航）

In [ ]:
# 噪聲旋轉矩陣的 QR 正交化
theta_true = 0.3
R_true = np.array([[np.cos(theta_true), -np.sin(theta_true)],
                   [np.sin(theta_true),  np.cos(theta_true)]])

# 加入量測噪聲
noise = np.random.default_rng(42).normal(0, 0.02, (2, 2))
R_noisy = R_true + noise

print(f'噪聲旋轉矩陣:')
print(f'  R^T R = \n{R_noisy.T @ R_noisy}')
print(f'  det = {np.linalg.det(R_noisy):.6f} (應為 1)')

# QR 正交化
Q, R = qr_decomposition(R_noisy)

print(f'\n正交化後:')
print(f'  Q^T Q = \n{np.round(Q.T @ Q, 10)}')
print(f'  det(Q) = {np.linalg.det(Q):.10f}')
print(f'  正交矩陣 ✓')

In [ ]:
# QR 解最小平方（比 Normal Equation 更穩定）
# Ax ≈ b → QRx = b → Rx = Q^T b
rng = np.random.default_rng(42)
m, n = 50, 3
A = rng.normal(size=(m, n))
x_true = np.array([1.5, -2.0, 3.0])
b = A @ x_true + rng.normal(0, 0.1, m)

# QR 方法
Q_qr, R_qr = qr_decomposition(A)
x_qr = sla.solve_triangular(R_qr, Q_qr.T @ b)

# Normal Equation
x_ne = np.linalg.solve(A.T @ A, A.T @ b)

# lstsq
x_lstsq, _, _, _ = np.linalg.lstsq(A, b, rcond=None)

print(f'真實值:       {x_true}')
print(f'QR 分解:      [{x_qr[0]:.4f}, {x_qr[1]:.4f}, {x_qr[2]:.4f}]')
print(f'Normal Eq:    [{x_ne[0]:.4f}, {x_ne[1]:.4f}, {x_ne[2]:.4f}]')
print(f'np.lstsq:     [{x_lstsq[0]:.4f}, {x_lstsq[1]:.4f}, {x_lstsq[2]:.4f}]')
print(f'三種方法一致 ✓')

---
## 5.3 Cholesky 分解

$A = LL^T$（對稱正定矩陣專用，效率是 LU 的兩倍）

### 物理應用：穩態熱傳導

In [ ]:
# 穩態熱傳導：左端 200°C, 右端 20°C
x_pos, T, K = heat_conduction_1d(n_nodes=10, T_left=200, T_right=20)

# Cholesky 分解導熱矩陣
L_chol = cholesky_decomposition(K)
L_np = np.linalg.cholesky(K)

print(f'手動 Cholesky 與 NumPy 一致: {np.allclose(L_chol, L_np)} ✓')
print(f'重組誤差: {np.linalg.norm(L_chol @ L_chol.T - K):.2e}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(x_pos, T, 'ro-', lw=2)
ax1.set_xlabel('位置 (m)')
ax1.set_ylabel('溫度 (°C)')
ax1.set_title('穩態溫度分佈')
ax1.grid(True, alpha=0.3)

plot_matrix_heatmap(L_chol, title='Cholesky L 矩陣', ax=ax2)
plt.tight_layout()
plt.show()

In [ ]:
# Cholesky vs LU 速度比較
sizes = [100, 200, 500, 1000]
times_chol = []
times_lu2 = []

for n in sizes:
    A = np.random.default_rng(42).normal(size=(n, n))
    A = A.T @ A + n * np.eye(n)

    t0 = time.perf_counter()
    for _ in range(10):
        np.linalg.cholesky(A)
    times_chol.append((time.perf_counter() - t0) / 10)

    t0 = time.perf_counter()
    for _ in range(10):
        sla.lu(A)
    times_lu2.append((time.perf_counter() - t0) / 10)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sizes, times_chol, 'go-', lw=2, label='Cholesky')
ax.plot(sizes, times_lu2, 'bo-', lw=2, label='LU')
ax.set_xlabel('矩陣大小 n')
ax.set_ylabel('平均時間 (秒)')
ax.set_title('Cholesky vs LU 分解速度')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

---
## 5.4 SVD — 奇異值分解（核心中的核心）

$A = U\Sigma V^T$

奇異值 = 資料的「重要性排序」

In [ ]:
# SVD 基礎
A = np.array([[1, 2], [3, 4], [5, 6]], dtype=float)
U, S, Vt = np.linalg.svd(A, full_matrices=False)

print(f'A ({A.shape}):')
print(A)
print(f'\nU ({U.shape}): 左奇異向量')
print(np.round(U, 4))
print(f'\nΣ = {np.round(S, 4)}: 奇異值')
print(f'\nV^T ({Vt.shape}): 右奇異向量')
print(np.round(Vt, 4))

# 重組驗證
A_reconstructed = U @ np.diag(S) @ Vt
print(f'\n重組誤差: {np.linalg.norm(A - A_reconstructed):.2e} < 1e-12 ✓')

In [ ]:
# SVD 圖像壓縮
# 建構一個模擬灰階影像（低秩結構 + 噪聲）
rng = np.random.default_rng(42)
m, n = 100, 80

# 建立有結構的影像
x = np.linspace(-3, 3, n)
y = np.linspace(-3, 3, m)
X, Y = np.meshgrid(x, y)
image = np.sin(X) * np.cos(Y) + 0.5 * np.sin(2*X) + rng.normal(0, 0.1, (m, n))

U, S, Vt = np.linalg.svd(image, full_matrices=False)

# 不同 k 值的壓縮
k_values = [1, 3, 5, 10, 20]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

axes[0].imshow(image, cmap='gray')
axes[0].set_title(f'原始 ({m}×{n})')
axes[0].axis('off')

for idx, k in enumerate(k_values):
    recon = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
    error = np.linalg.norm(image - recon) / np.linalg.norm(image)
    compression = (k * (m + n + 1)) / (m * n)
    axes[idx+1].imshow(recon, cmap='gray')
    axes[idx+1].set_title(f'k={k}, 誤差={error:.3f}\n壓縮率={compression:.2%}')
    axes[idx+1].axis('off')

plt.suptitle('SVD 影像壓縮', fontsize=14)
plt.tight_layout()
plt.show()

# 奇異值衰減圖
fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(S, 'bo-')
ax.set_xlabel('奇異值索引')
ax.set_ylabel('奇異值大小 (log)')
ax.set_title('奇異值衰減曲線')
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 機器手臂 Jacobian SVD — 可操作性橢球
theta1_val, theta2_val = np.pi/4, np.pi/6
L1, L2 = 1.0, 0.8

J = np.array([
    [-L1*np.sin(theta1_val) - L2*np.sin(theta1_val+theta2_val),
     -L2*np.sin(theta1_val+theta2_val)],
    [L1*np.cos(theta1_val) + L2*np.cos(theta1_val+theta2_val),
     L2*np.cos(theta1_val+theta2_val)],
])

U_j, S_j, Vt_j = np.linalg.svd(J)

print(f'Jacobian 奇異值: σ₁={S_j[0]:.4f}, σ₂={S_j[1]:.4f}')
print(f'可操作性 (σ_min/σ_max) = {S_j[1]/S_j[0]:.4f}')
print(f'可操作性指標 (Σσ) = {np.prod(S_j):.4f}')

# 繪製可操作性橢球
fig, ax = plt.subplots(figsize=(8, 8))
theta_plot = np.linspace(0, 2*np.pi, 100)
# 橢球 = U @ diag(σ) @ 單位圓
circle = np.array([np.cos(theta_plot), np.sin(theta_plot)])
ellipse = U_j @ np.diag(S_j) @ circle

# 末端位置
end_x = L1*np.cos(theta1_val) + L2*np.cos(theta1_val+theta2_val)
end_y = L1*np.sin(theta1_val) + L2*np.sin(theta1_val+theta2_val)

ax.plot(ellipse[0]+end_x, ellipse[1]+end_y, 'b-', lw=2)
ax.fill(ellipse[0]+end_x, ellipse[1]+end_y, alpha=0.2, color='blue')
# 主軸
for i in range(2):
    ax.annotate('', xy=(end_x+U_j[0,i]*S_j[i], end_y+U_j[1,i]*S_j[i]),
                xytext=(end_x, end_y),
                arrowprops=dict(arrowstyle='->', color='red', lw=2))
    ax.text(end_x+U_j[0,i]*S_j[i]*1.1, end_y+U_j[1,i]*S_j[i]*1.1,
            f'σ{i+1}={S_j[i]:.2f}', fontsize=11, color='red')

# 機器手臂
ax.plot([0, L1*np.cos(theta1_val)], [0, L1*np.sin(theta1_val)], 'k-o', lw=3)
ax.plot([L1*np.cos(theta1_val), end_x], [L1*np.sin(theta1_val), end_y], 'k-o', lw=3)
ax.plot(end_x, end_y, 'g*', ms=15)

ax.set_aspect('equal')
ax.set_title('機器手臂可操作性橢球')
ax.grid(True, alpha=0.3)
plt.show()

---
## Module 5 驗證總結

| 項目 | 驗證方式 | 結果 |
|------|----------|------|
| LU: L@U = A | 重組誤差 < 1e-12 | ✓ |
| QR: Q^TQ = I | 正交性 | ✓ |
| Cholesky: LL^T = A | 重組誤差 < 1e-12 | ✓ |
| SVD: UΣV^T = A | 重組誤差 < 1e-12 | ✓ |
| SVD 壓縮 | 視覺品質 | ✓ |
| 手動 vs NumPy | np.allclose | ✓ |